In [6]:
import pandas as pd
import numpy as np

# Set seed for reproducible results
np.random.seed(42)

n_samples = 500

# 1. Base Demographics
customer_ids = [f"CUST_{1000 + i}" for i in range(n_samples)]

# Age with continuous uniform measurement noise
age = np.random.normal(loc=35, scale=10, size=n_samples)
age += np.random.uniform(-1.5, 1.5, size=n_samples)
age = np.clip(age, 18, 68).round(1)

# Categorical Education
education_levels = ['High School', 'Bachelor', 'Master', 'PhD']
education = np.random.choice(education_levels, size=n_samples, p=[0.25, 0.45, 0.20, 0.10])

# --- NOISE INJECTED: Inconsistent casing & Whitespace ---
edu_noisy = education.copy().astype(object)
for i in range(n_samples):
    r = np.random.rand()
    if r < 0.04:
        edu_noisy[i] = edu_noisy[i].lower()           # e.g., 'bachelor'
    elif r < 0.07:
        edu_noisy[i] = f"  {edu_noisy[i]} "          # e.g., '  High School '

# Trend 1: Experience with random error/variance
years_exp = np.maximum(0, age - 22 + np.random.normal(0, 3.5, n_samples)).round(1)

# Trend 2: Income with higher standard deviation/variance
edu_base_income = {'High School': 32, 'Bachelor': 48, 'Master': 62, 'PhD': 80}
base_inc = np.array([edu_base_income[e] for e in education])
annual_income = base_inc + (years_exp * 1.8) + np.random.normal(0, 12, n_samples)
annual_income = np.round(np.clip(annual_income, 18, 190), 2)

# Categorical Device Type with string noise
devices = ['Mobile', 'Desktop', 'Tablet']
device_type = np.random.choice(devices, size=n_samples, p=[0.55, 0.35, 0.10])
device_noisy = device_type.copy().astype(object)
for i in range(n_samples):
    if np.random.rand() < 0.04:
        device_noisy[i] = device_noisy[i].lower()

# Trend 3: Is_Premium ('Yes'/'No') with 10% outcome flip noise
membership_prob = 1 / (1 + np.exp(-(annual_income - 65) / 15))
raw_premium = np.random.binomial(1, membership_prob)
flip_mask = np.random.rand(n_samples) < 0.10  # 10% random flip noise
raw_premium_flipped = np.where(flip_mask, 1 - raw_premium, raw_premium)
is_premium = np.where(raw_premium_flipped == 1, 'Yes', 'No')

# Trend 4: Session duration with wider spread
session_means = {'Mobile': 12, 'Desktop': 28, 'Tablet': 18}
session_duration = np.array([
    np.random.normal(session_means[d], 7) for d in device_type
]).round(1)
session_duration = np.clip(session_duration, 0.5, 75.0)

# Trend 5: Total Spend (USD) with heavy Gaussian noise
spend_noise = np.random.normal(0, 250, n_samples)
premium_numeric = (is_premium == 'Yes').astype(int)
total_spend = (annual_income ** 1.35) * 0.8 + (premium_numeric * 350) + (session_duration * 12) + spend_noise
total_spend = np.round(np.clip(total_spend, 20, None), 2)

# Trend 6: Satisfied ('Yes'/'No') with classification boundary noise
spend_per_min = total_spend / session_duration
satisfaction_prob = 1 / (1 + np.exp(-(spend_per_min - 30) / 20))
raw_sat = np.random.binomial(1, satisfaction_prob)
flip_sat = np.random.rand(n_samples) < 0.08  # 8% label noise
raw_sat_flipped = np.where(flip_sat, 1 - raw_sat, raw_sat)
satisfied = np.where(raw_sat_flipped == 1, 'Yes', 'No')

# Construct DataFrame
df = pd.DataFrame({
    'Customer_ID': customer_ids,
    'Age': age,
    'Education': edu_noisy,
    'Years_Experience': years_exp,
    'Annual_Income_k$': annual_income,
    'Device_Type': device_noisy,
    'Is_Premium': is_premium,
    'Avg_Session_Min': session_duration,
    'Total_Spend_USD': total_spend,
    'Satisfied': satisfied
})

# --- INJECT REAL-WORLD NOISE & IMPERFECTIONS ---

# 1. Missing Values (NaNs across multiple columns)
df.loc[np.random.rand(n_samples) < 0.06, 'Annual_Income_k$'] = np.nan
df.loc[np.random.rand(n_samples) < 0.05, 'Avg_Session_Min'] = np.nan
df.loc[np.random.rand(n_samples) < 0.04, 'Years_Experience'] = np.nan

# Save dataset
df.to_csv('eda_practice_noisy_dataset.csv', index=False)

print("Dataset generated successfully with shape:", df.shape)
print("\nFirst 10 rows:")
df.head()

Dataset generated successfully with shape: (500, 10)

First 10 rows:


,Customer_ID,Age,Education,Years_Experience,Annual_Income_k$,Device_Type,Is_Premium,Avg_Session_Min,Total_Spend_USD,Satisfied
0,CUST_1000,40.4,Master,24.1,95.36,Mobile,Yes,12.2,1273.53,Yes
1,CUST_1001,33.8,High School,10.7,76.33,Mobile,Yes,1.2,571.56,Yes
2,CUST_1002,41.0,Master,23.3,84.65,Mobile,Yes,13.2,863.24,Yes
3,CUST_1003,51.7,High School,35.0,97.22,Mobile,Yes,NaN,793.24,No
4,CUST_1004,33.0,Bachelor,14.5,98.38,Mobile,Yes,15.5,711.14,No


In [8]:
df.describe()

,Age,Years_Experience,Annual_Income_k$,Avg_Session_Min,Total_Spend_USD
count,500.000000,483.000000,464.000000,472.000000,500.000000
mean,35.184400,13.471014,74.131444,18.134534,714.307700
std,9.545898,9.556996,25.545659,9.564374,360.843149
min,18.000000,0.000000,18.000000,0.500000,20.000000
25%,28.275000,5.800000,56.025000,11.300000,469.312500
50%,35.400000,12.700000,73.525000,17.400000,726.120000
75%,41.300000,20.600000,92.260000,25.200000,961.235000
max,68.000000,44.900000,158.440000,44.700000,1735.960000


In [9]:
df.head()

,Customer_ID,Age,Education,Years_Experience,Annual_Income_k$,Device_Type,Is_Premium,Avg_Session_Min,Total_Spend_USD,Satisfied
0,CUST_1000,40.4,Master,24.1,95.36,Mobile,Yes,12.2,1273.53,Yes
1,CUST_1001,33.8,High School,10.7,76.33,Mobile,Yes,1.2,571.56,Yes
2,CUST_1002,41.0,Master,23.3,84.65,Mobile,Yes,13.2,863.24,Yes
3,CUST_1003,51.7,High School,35.0,97.22,Mobile,Yes,NaN,793.24,No
4,CUST_1004,33.0,Bachelor,14.5,98.38,Mobile,Yes,15.5,711.14,No


In [ ]:
df.corr(numeric_only=True)
# Years of experience is positively correlated with Income
# Annual income is fairly correlated with age maybe because more age potentiaties to more expericence
# Annual income is fairly correlated with Total Spend
# Total Spend is corr with Annual income
# No major negative correlation

# Some minor correlation


,Age,Years_Experience,Annual_Income_k$,Avg_Session_Min,Total_Spend_USD
Age,1.000000,0.943096,0.668447,-0.067146,0.354529
Years_Experience,0.943096,1.000000,0.700276,-0.052886,0.384100
Annual_Income_k$,0.668447,0.700276,1.000000,-0.017682,0.551915
Avg_Session_Min,-0.067146,-0.052886,-0.017682,1.000000,0.319985
Total_Spend_USD,0.354529,0.384100,0.551915,0.319985,1.000000


In [12]:
df.head()

,Customer_ID,Age,Education,Years_Experience,Annual_Income_k$,Device_Type,Is_Premium,Avg_Session_Min,Total_Spend_USD,Satisfied
0,CUST_1000,40.4,Master,24.1,95.36,Mobile,Yes,12.2,1273.53,Yes
1,CUST_1001,33.8,High School,10.7,76.33,Mobile,Yes,1.2,571.56,Yes
2,CUST_1002,41.0,Master,23.3,84.65,Mobile,Yes,13.2,863.24,Yes
3,CUST_1003,51.7,High School,35.0,97.22,Mobile,Yes,NaN,793.24,No
4,CUST_1004,33.0,Bachelor,14.5,98.38,Mobile,Yes,15.5,711.14,No


In [ ]:
df["Education"].value_counts(normalize = True)
# Majority of users are on mobile closely followed by desktop
# 61 % users are premium, 39 percent are not
# only 57% Satisfaction rate !
# Majority users are bachelor followed by high school



Education
Bachelor          0.422
High School       0.246
Master            0.166
PhD               0.092
  Bachelor        0.016
  Master          0.014
master            0.014
high school       0.010
  High School     0.008
bachelor          0.008
phd               0.002
  PhD             0.002
Name: proportion, dtype: float64

In [19]:
df.head()

,Customer_ID,Age,Education,Years_Experience,Annual_Income_k$,Device_Type,Is_Premium,Avg_Session_Min,Total_Spend_USD,Satisfied
0,CUST_1000,40.4,Master,24.1,95.36,Mobile,Yes,12.2,1273.53,Yes
1,CUST_1001,33.8,High School,10.7,76.33,Mobile,Yes,1.2,571.56,Yes
2,CUST_1002,41.0,Master,23.3,84.65,Mobile,Yes,13.2,863.24,Yes
3,CUST_1003,51.7,High School,35.0,97.22,Mobile,Yes,NaN,793.24,No
4,CUST_1004,33.0,Bachelor,14.5,98.38,Mobile,Yes,15.5,711.14,No


In [39]:
pd.crosstab(df["Education"], df["Is_Premium"], normalize="index") * 100 # Normalise =  index means each row sums to 100

# 70% of satisfied users are premium users
# 75% of high schoolers are not premium while others are mostly 50 - 50

Is_Premium,No,Yes
Education,,
Bachelor,50.000000,50.000000
High School,75.000000,25.000000
Master,42.857143,57.142857
PhD,0.000000,100.000000
Bachelor,42.654028,57.345972
High School,51.219512,48.780488
Master,28.915663,71.084337
PhD,8.695652,91.304348
bachelor,0.000000,100.000000


In [35]:
df.head()

,Customer_ID,Age,Education,Years_Experience,Annual_Income_k$,Device_Type,Is_Premium,Avg_Session_Min,Total_Spend_USD,Satisfied
0,CUST_1000,40.4,Master,24.1,95.36,Mobile,Yes,12.2,1273.53,Yes
1,CUST_1001,33.8,High School,10.7,76.33,Mobile,Yes,1.2,571.56,Yes
2,CUST_1002,41.0,Master,23.3,84.65,Mobile,Yes,13.2,863.24,Yes
3,CUST_1003,51.7,High School,35.0,97.22,Mobile,Yes,NaN,793.24,No
4,CUST_1004,33.0,Bachelor,14.5,98.38,Mobile,Yes,15.5,711.14,No


In [43]:
df.pivot_table(
    index = "Device_Type",
    values= "Avg_Session_Min",
    aggfunc = ["mean", "median"]
)

,mean,median
,Avg_Session_Min,Avg_Session_Min
Device_Type,,
Desktop,27.036686,27.30
Mobile,12.196234,12.90
Tablet,16.902174,16.75
desktop,24.942857,24.70
mobile,11.280000,9.60
tablet,10.500000,10.50


In [ ]:
Final Observations:

General:
More year of experience yields more income
More the annual income more the spending
Widely used by bachelors

Critical:
The net satisfation rate is only 57%
70% of satisfied users are premium users
75% of high schoolers are not premium while others are mostly 50 - 50


Skeptical (Not complete data):
91.3 of PhD are premium subscribers

